
# SIAM Demo Notebook
This notebook demonstrates a minimal end-to-end workflow using the **synthetic** `example_dataset.csv`:
- Load data
- Split train/test
- Train a simple Random Forest classifier
- Evaluate with accuracy, confusion matrix, and ROC curve

> **Note:** The dataset is synthetic and intended **only** for demonstration.


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42


In [ ]:

# Path to the example dataset (adjust if needed)
DATA_PATH = '/mnt/data/example_dataset.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()


In [ ]:

# Separate features and labels
X = df.drop(columns=['clase', 'archivo'])
y = df['clase'].map({'ALZ':1, 'CONTROL':0})  # binary encoding

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

X_train.shape, X_test.shape


In [ ]:

# Simple Random Forest model
clf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=None, 
    random_state=RANDOM_STATE
)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f'Accuracy: {acc:.3f}')
print(f'ROC AUC:  {auc:.3f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['CONTROL','ALZ']))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))


In [ ]:

# ROC curve
fpr, tpr, thr = roc_curve(y_test, y_prob)
plt.figure()
plt.plot(fpr, tpr, label=f'RandomForest (AUC={auc:.2f})')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()
